<a href="https://colab.research.google.com/github/samilnamli/transformer_experiments/blob/colab/voxpopuli-main-results/notebooks/colab/main_results_voxpopuli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VoxPopuli — Whisper correction (Colab)

Runs the **Whisper re-decode** that fixes VoxPopuli's broken Whisper WER
(raw ~121% → ~18% median), and — optionally — the full main-results
comparison table.

Uses a Blackwell-compatible torch (cu128) so it runs on the RTX PRO 6000.
Run the cells top to bottom.

## 1. Setup — clone & install

In [1]:
%cd /content
!pip -q install uv
!git clone -b colab/voxpopuli-main-results https://github.com/samilnamli/transformer_experiments.git
%cd /content/transformer_experiments
!uv sync
!uv run python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0))"

/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 154.4 MB/s eta 0:00:00
Cloning into 'transformer_experiments'...
remote: Enumerating objects: 1419, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 1419 (delta 43), reused 55 (delta 19), pack-reused 1317 (from 2)
Receiving objects: 100% (1419/1419), 740.23 MiB | 27.42 MiB/s, done.
Resolving deltas: 100% (697/697), done.
/content/transformer_experiments
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Resolved 272 packages in 1.17s
Prepared 265 packages in 23.49s
Installed 265 packages in 393ms
 + absl-py==2.4.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + alembic==1.18.4
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.13.0
 + appdirs==1.4.4
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + asy

## 2. HuggingFace token

Add `HF_TOKEN` under Colab **🔑 Secrets** (left sidebar).

In [2]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

## 3. Download data (~24 GB)

In [5]:
import os
import gdown

# Define the target directory and file path
target_dir = "/content/transformer_experiments/configs/data/processed/facebook_voxpopuli"
os.makedirs(target_dir, exist_ok=True)

out_path = os.path.join(target_dir, "combined_features_with_transcripts.parquet")

# Download the file using gdown
print(f"Downloading to {out_path}...")
gdown.download(id="1yf-G-DWhhZLlqeGXZ77GbuhTBmhqyyXA", output=out_path, quiet=False)

# Verify the file existence and size
if os.path.exists(out_path):
    size_gb = os.path.getsize(out_path) / (1024**3)
    print(f"\nSuccess! File exists.")
    print(f"Size: {size_gb:.2f} GB")
    !ls -lh {target_dir}
else:
    print("\nError: Download failed or file not found.")

Downloading...
From (original): https://drive.google.com/uc?id=1yf-G-DWhhZLlqeGXZ77GbuhTBmhqyyXA
From (redirected): https://drive.google.com/uc?id=1yf-G-DWhhZLlqeGXZ77GbuhTBmhqyyXA&confirm=t&uuid=31411016-4d14-4304-952b-7def0613ad12
To: /content/transformer_experiments/configs/data/processed/facebook_voxpopuli/combined_features_with_transcripts.parquet
100%|██████████| 24.2G/24.2G [04:10<00:00, 96.5MB/s]


Success! File exists.
Size: 22.55 GB
total 23G
-rw-r--r-- 1 root root 23G Apr 30 07:14 combined_features_with_transcripts.parquet


## 4. Whisper correction (re-decode)

Writes the corrected overrides `.npz`. Takes ~1 h on the GPU.

In [6]:
!uv run python scripts/redecode_voxpopuli_whisper_overrides_from_features.py \
  --input-parquet configs/data/processed/facebook_voxpopuli/combined_features_with_transcripts.parquet \
  --output-npz data/processed/facebook_voxpopuli/whisper_decode_overrides_from_features.npz

Streaming output truncated to the last 5000 lines.
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/

In [8]:
import os
from huggingface_hub import HfApi

REPO = 'huseyin-karaca/hit-asr'
FILES = {
    'configs/data/processed/facebook_voxpopuli/combined_features_with_transcripts.parquet':
        'combined_features_with_transcripts.parquet',
    'data/processed/facebook_voxpopuli/whisper_decode_overrides_from_features.npz':
        'whisper_decode_overrides_from_features.npz',
}
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(REPO, repo_type='dataset', exist_ok=True)
for local, name in FILES.items():
    print('uploading', name)
    api.upload_file(path_or_fileobj=local, path_in_repo=name,
                    repo_id=REPO, repo_type='dataset')
print('done ->', 'https://huggingface.co/datasets/' + REPO)

# Future runs — replace sections 3+4 with this to skip Drive + re-decode:
#   from huggingface_hub import hf_hub_download
#   hf_hub_download(REPO, 'combined_features_with_transcripts.parquet', repo_type='dataset',
#                   local_dir='configs/data/processed/facebook_voxpopuli')
#   hf_hub_download(REPO, 'whisper_decode_overrides_from_features.npz', repo_type='dataset',
#                   local_dir='data/processed/facebook_voxpopuli')

uploading combined_features_with_transcripts.parquet


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._with_transcripts.parquet:   0%|          |  524kB / 24.2GB            

uploading whisper_decode_overrides_from_features.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...errides_from_features.npz:  96%|#########6|  524kB /  544kB            

done -> https://huggingface.co/datasets/huseyin-karaca/hit-asr


## 5. (Optional) Full comparison table

20 fun seeds by default. Trim with `experiment.seeds` for a quick pass.
Aggregation + significance tests run automatically at the end.

In [10]:
!uv run python run.py experiment=main_results_voxpopuli_colab \
  experiment.data.batch_size=512 \
  experiment.seeds=[42,1337,73] \
  experiment.data.eager_load=true

/content/transformer_experiments/.venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026-07-28 08:21:56 INFO     src.utils.mlflow_setup:setup_mlflow:57 | mlflow: tracking_uri=mlruns experiment=main_results_voxpopuli_colab
2026-07-28 08:21:57 INFO     torch.distributed.nn.jit.instantiator:<module>:24 | Created a temporary directory at /tmp/tmplbvc8ha7
2026-07-28 08:21:57 INFO     torch.distributed.nn.jit.instantiator:_write:75 | Writing /tmp/tmplbvc8ha7/_remote_module_non_scriptable.py
2026-07-28 08:21:58 INFO     lightning_fabric.utilities.seed:seed_everything:57 | Seed set to 42
2026-07-28 08:21:58

In [ ]:
!uv run python run.py experiment=main_results_voxpopuli_colab \
  experiment.data.batch_size=512 \
  experiment.seeds=[1729, 8675309, 31415, 27182, 16180, 112358, 6022
                    #, 6626, 65536, 2048, 9001, 8086, 12321, 86400, 299792, 90210, 404
                    ] \
  experiment.data.eager_load=true